# পাঠ ০৮ - বহু-এজেন্ট ডিজাইন প্যাটার্ন


## সেটআপ


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## কেন মাল্টি-এজেন্ট সিস্টেম?

বাস্তব জীবনের কাজ যেমন ট্রিপ পরিকল্পনা বিভিন্ন ধরনের দক্ষতা জড়িত — লজিস্টিক, স্থানীয় জ্ঞান, বাজেটিং, এবং আরও অনেক কিছু। সবকিছু সামলাতে একক এজেন্ট দ্রুত ঝামেলাপূর্ণ হয়ে যায়।

মাল্টি-এজেন্ট সিস্টেমগুলি এটি সমাধান করে **বৈশিষ্ট্যকরণ** এর মাধ্যমে: প্রতিটি এজেন্ট একক দক্ষতার উপর মনোনিবেশ করে, যার ফলে সাধারণজ্ঞের তুলনায় উচ্চ-মানের ফলাফল করে। তারা **স্কেলেবিলিটি** বৃদ্ধি করে — আপনি নতুন এজেন্ট যোগ করতে পারেন (যেমন, একটি ফ্লাইট বিশেষজ্ঞ, একটি রেস্তোরাঁ সমালোচক) বিদ্যমান কর্মপ্রবাহ পুনর্লিখন ছাড়া। এজেন্টগুলি একটি কাঠামোবদ্ধ পাইপলাইনের মাধ্যমে একত্রিত হয়, প্রেক্ষাপট এক থেকে আরেকটিতে স্থানান্তর করে।


## বিশেষায়িত এজেন্ট তৈরি করা


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## একটি ধারাবাহিক ওয়ার্কফ্লো তৈরি করা

`WorkflowBuilder` আপনাকে এজেন্টদের একটি পরিচালিত গ্রাফে সংযুক্ত করার সুযোগ দেয়। এখানে আমরা একটি সাধারণ দুই-ধাপের পাইপলাইন তৈরি করি: **TravelPlanner** একটি অভিযানসূচী খসড়া করে, তারপর **TravelConcierge** এটি পর্যালোচনা ও উন্নত করে।


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ওয়ার্কফ্লোতে আরও এজেন্ট যোগ করা হচ্ছে

মাল্টি-এজেন্ট প্যাটার্নের সবচেয়ে বড় সুবিধা হলো এটি কত সহজে সম্প্রসারিত করা যায়। নিচে আমরা একটি **BudgetReviewer** এজেন্ট যোগ করেছি যা পরিকল্পনাটি ভ্রমণকারীর বাজেটের বিপরীতে পরীক্ষা করে, সীমার বাইরে খরচ বাড়াতে পারে এমন আইটেমগুলিকে চিহ্নিত করে, এবং টাকা সাশ্রয়ের বিকল্প পরামর্শ দেয়। এখন ওয়ার্কফ্লো তিনটি এজেন্ট ক্রমান্বয়ে চালায়:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## সারাংশ

এই পাঠে আপনি শিখেছেন কীভাবে:

1. **বিশেষায়িত এজেন্ট তৈরি করতে হয়** — প্রত্যেকটির একটি কেন্দ্রীভূত ভূমিকা থাকে (পরিকল্পনা, কনসিয়ার্জ, বাজেট পর্যালোচনা)।
2. **এজেন্টগুলোকে ক্রমানুসারে ওয়ার্কফ্লোতে যুক্ত করতে হয়** `WorkflowBuilder` এবং `add_edge` ব্যবহার করে।
3. **মাল্টি-এজেন্ট পাইপলাইনের আউটপুট স্ট্রিম করতে হয়**, ট্র্যাক করা হয় কোন এজেন্টটি কথা বলছে।
4. **একটি ওয়ার্কফ্লো সম্প্রসারিত করতে হয়** নতুন এজেন্ট যোগ করে চেইনে, বিদ্যমানগুলো সংশোধন না করেই।

মাল্টি-এজেন্ট ডিজাইন প্যাটার্ন প্রতিটি এজেন্টকে সহজ রাখে, একই সঙ্গে আরও সমৃদ্ধ এবং ভালোভাবে পর্যালোচিত ফলাফল তৈরি করে যা কোনো একক এজেন্ট একা পারত না।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**দ্রষ্টব্য**:  
এই দলিলটি AI অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। আমরা যথাসাধ্য সঠিকতার জন্য সচেষ্ট হলেও, স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল দলিলের স্থানীয় ভাষার সংস্করণকে কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচনা করা উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানুষের দ্বারা অনুবাদ করানোকে সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে কোনও ভুল বোঝাবুঝি বা ব্যত্যয়ের জন্য আমরা দায়ী থাকব না।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
